The two methods you’re seeing now — **explode()** and **assign()** — are deceptively simple but incredibly powerful once you see how they fit into the broader reshaping toolkit you’ve been building (stack/unstack, swaplevel, sort_index, wide_to_long, MultiIndex intuition, etc.).

They’re used constantly in real data pipelines, especially when cleaning messy, nested, or semi structured data.


In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.DataFrame({
    "name": ["Alice", "Bob"],
    "hobbies": [["reading", "cycling"], ["gaming"]]
})
df

,name,hobbies
0,Alice,"[reading, cycling]"
1,Bob,[gaming]


In [3]:
df.explode("hobbies")

,name,hobbies
0,Alice,reading
0,Alice,cycling
1,Bob,gaming


In [4]:
df.assign(
    name_upper = df["name"].str.upper(),
    hobby_count = df["hobbies"].str.len()
)

,name,hobbies,name_upper,hobby_count
0,Alice,"[reading, cycling]",ALICE,2
1,Bob,[gaming],BOB,1


### My Question:
The example "df.assign( name_upper = df["name"].str.upper(), hobby_count = df["hobbies"].str.len() )" is a bit confusing for hobby_count. It figures out the number of elements in the list ...Why did we need the str ? why not just use .len()??

You’re absolutely right to pause here — df["hobbies"].str.len() looks odd when the values are Python lists, and your instinct (“why not just use .len()?”) is exactly the right question.
The key is this:
.str is not only for strings — it is the “vectorized accessor” for any sequence‑like object stored in a Series.

It lets pandas apply Python’s len() to each element element‑wise, safely and consistently.

Why str.len() works on lists
Even though the column is called "hobbies" and contains lists, pandas treats .str as a general-purpose accessor for sequence operations.
- If the element is a string → .str.len() returns number of characters
- If the element is a list → .str.len() returns number of items
- If the element is a tuple → .str.len() returns number of items
- If the element is a set → .str.len() returns number of items
So .str.len() is really:
“Apply Python’s len() to each element of the Series.”


Why you cannot use .len() directly
If you try:

```
df["hobbies"].len()
```
You get the length of the Series (number of rows), not the length of each list.
If you try:
```
df["hobbies"].apply(len)
```
This works — but .apply() is slower and less idiomatic than .str.len().

In [5]:
df.assign(
    name_upper = df["name"].str.upper(),
    name_length = lambda d: d["name_upper"].str.len()
)

,name,hobbies,name_upper,name_length
0,Alice,"[reading, cycling]",ALICE,5
1,Bob,[gaming],BOB,3


In [6]:
df = pd.DataFrame({
    "order_id": [1, 2],
    "items": [["apple", "banana"], ["banana", "pear"]],
    "customer": ["Lorraine", "Lorraine"]
})
df

,order_id,items,customer
0,1,"[apple, banana]",Lorraine
1,2,"[banana, pear]",Lorraine


In [9]:
df1 = df.explode("items")
df1

,order_id,items,customer
0,1,apple,Lorraine
0,1,banana,Lorraine
1,2,banana,Lorraine
1,2,pear,Lorraine


In [10]:
# assign a new column
df2 = df1.assign(
    item_length = df1["items"].str.len()
)
df2

,order_id,items,customer,item_length
0,1,apple,Lorraine,5
0,1,banana,Lorraine,6
1,2,banana,Lorraine,6
1,2,pear,Lorraine,4


In [11]:
df2.groupby("items")["order_id"].count()

items
apple     1
banana    2
pear      1
Name: order_id, dtype: int64

In [12]:
# example with MultiIndex
df = pd.DataFrame({
    "city": ["NY", "NY", "LA"],
    "year": [2023, 2024, 2023],
    "temps": [[30, 32, 35], [28, 29], [40, 42]]
}).set_index(["city", "year"])
df

temps
city year              
NY   2023  [30, 32, 35]
     2024      [28, 29]
LA   2023      [40, 42]

In [13]:
#explode
df_ex = df.explode("temps")
df_ex

temps
city year      
NY   2023    30
     2023    32
     2023    35
     2024    28
     2024    29
LA   2023    40
     2023    42

In [14]:
# assign a temperature category
df_ex = df_ex.assign(
    temp_category = lambda d: pd.cut(
        d["temps"],
        bins=[0, 32, 40, 100],
        labels=["cold", "warm", "hot"]
    )
)
df_ex

temps temp_category
city year                    
NY   2023    30          cold
     2023    32          cold
     2023    35          warm
     2024    28          cold
     2024    29          cold
LA   2023    40          warm
     2023    42           hot

In [16]:
# group by MultiIndex
df_ex.groupby(["city", "temp_category"], observed=False).size()

city  temp_category
LA    cold             0
      warm             1
      hot              1
NY    cold             4
      warm             1
      hot              0
dtype: int64

In [17]:
# explode + assign + unstack
df = pd.DataFrame({
    "student": ["A", "B"],
    "scores": [[80, 90], [70, 75]]
})
df

,student,scores
0,A,"[80, 90]"
1,B,"[70, 75]"


In [19]:
# explode
df1 = df.explode("scores")
df1

,student,scores
0,A,80
0,A,90
1,B,70
1,B,75


In [20]:
# assign exam number
df2 = df1.assign(
    exam = df1.groupby("student").cumcount() + 1
)
df2

,student,scores,exam
0,A,80,1
0,A,90,2
1,B,70,1
1,B,75,2


In [21]:
# unstack
df2.set_index(["student", "exam"]).unstack()

scores    
exam         1   2
student           
A           80  90
B           70  75